# Route 97 delay congestion and propagation

Estimate congestion from **delay gained or recovered while traversing each directed stop-to-stop segment**, rather than bus speed or total lateness. For the same trip instance, `segment delay change = downstream arrival_delay - upstream arrival_delay`.

- **Positive:** the segment added delay.
- **Zero:** no additional delay, even if the bus was already late.
- **Negative:** delay recovery relative to schedule.
- **Missing:** unknown, not zero congestion.

The latest reported arrival-delay estimate is an approved **ground-truth proxy**, not an independently measured arrival. Arrival-to-arrival differences include upstream dwell, holding, and schedule slack, so the result is a delay-accumulation traffic proxy, not proof of pure road congestion.

## Requirements and scope

Run with the `transit_env` Conda environment. Install the existing notebook dependencies from the repository with `cd forecasting && pip install -r requirements.txt` if needed.

This notebook keeps the original analysis scope: bus route **97** (`97-KEL` in GTFS), direction **0**, five-minute UTC bins, and a selected saved backtest. Choose the snapshot explicitly with `backtest_dataset_id` below. Cache mode makes no database or download calls and uses the snapshot's static GTFS. Plotly maps may fetch base-map tiles when opened in a browser.

Reusable selection, geometry, pairing, aggregation, and replay logic lives in `forecasting`; the original speed-based notebook is unchanged.

In [ ]:
from dataclasses import replace
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from forecasting import (
    compute_delay_anomalies,
    event_space_time_response,
    get_backtest_dataset,
    list_backtest_datasets,
    make_delay_congestion_map,
    make_delay_event_plot,
    make_propagation_heatmap,
    make_time_space_diagram,
    order_route_segments,
    prepare_delay_congestion_snapshots,
    run_delay_congestion_engine,
    same_bus_propagation,
    summarize_propagation,
)

In [ ]:
workspace_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
repo_root = next(
    (candidate for candidate in workspace_candidates
     if (candidate / 'forecasting' / 'forecasting').is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository forecasting package.')

backtest_cache_dir = repo_root / 'data' / 'backtests'
visuals_dir = repo_root / 'notebooks' / 'visuals'
visuals_dir.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo_root}')

## Download backtest data

Reuse the original notebook's `get_backtest_dataset` workflow and selected snapshot:

- **`'cache'`** (default): replay the specified dataset offline. A missing/corrupt snapshot raises an error; it never triggers a replacement download. Only load trusted pickle snapshots.
- **`'download'`**: explicitly download a new realtime window and current static GTFS, saving a snapshot named by `backtest_dataset_id`. Set it to `None` for an automatic UTC timestamp name. An occupied name raises `FileExistsError` before credentials are read or any database/network calls are made. This mode alone reads `.env.road_segment_congestion` or `TRANSIT_DB_*` credentials. Source `./setup_road_segment_congestion_env.sh` from the repository root to configure credentials if needed.

Changing the dataset is an explicit choice. Catalog order is save time, not data coverage. Cached analyses always use the selected snapshot's own static GTFS.

Download SQL below selects Route `97-KEL`, direction `0`, on the database before transferring either table. Each query uses named placeholders and its own parameter dictionary: `vehicle_query_params` or `trip_query_params`. Set `lookback_hours` in each dictionary to `7 * 24` for the latest week, and adjust `route_id` and `direction_id` there as needed. The dictionaries can have different values or keys when the SQL differs. Set `vehicle_query` or `trip_query` to `None` to restore the default all-route query using that dictionary's `lookback_hours`. SQL and parameter overrides only affect downloads; cache mode replays the selected snapshot unchanged.

In [ ]:
display(list_backtest_datasets(backtest_cache_dir))

In [ ]:
backtest_source = 'cache'
#backtest_source = 'download'
# Cache: select an existing ID. Download: choose an unused ID, or None for automatic naming.
backtest_dataset_id = "2026-11-10_2_months"
route_id = '97-KEL'
trip_direction_id = 0

vehicle_query = """
SELECT *
FROM TRANSIT_VEHICLE_TABLE
WHERE feed_timestamp >= NOW() - INTERVAL %(lookback_hours)s HOUR
  AND trip_route_id = %(route_id)s
  AND trip_direction_id = %(direction_id)s
ORDER BY id DESC
"""

trip_query = """
SELECT *
FROM TRANSIT_TRIP_TABLE
WHERE feed_timestamp >= NOW() - INTERVAL %(lookback_hours)s HOUR
  AND trip_route_id = %(route_id)s
  AND trip_direction_id = %(direction_id)s
ORDER BY id DESC
"""

query_params = {
    'lookback_hours': 2*30*24,
    'route_id': route_id,
    'direction_id': trip_direction_id,
}


backtest_dataset = get_backtest_dataset(
    repo_root / 'data',
    source=backtest_source,
    dataset_id=backtest_dataset_id,
    env_file=repo_root / '.env.road_segment_congestion',
    vehicle_query=vehicle_query,
    vehicle_query_params=query_params,
    trip_query=trip_query,
    trip_query_params=query_params,
)
df_vehicle_updates = backtest_dataset.vehicle_updates
df_trip_updates = backtest_dataset.trip_updates
static_data_dir = backtest_dataset.static_data_dir
print(f'Using backtest dataset: {backtest_dataset.directory.name}')
print(f'Vehicle/trip updates: {len(df_vehicle_updates):,} / {len(df_trip_updates):,}')
print(f'Trip-update UTC window: {df_trip_updates.feed_timestamp.min()} through {df_trip_updates.feed_timestamp.max()}')

In [ ]:
id_dtypes = {'stop_id': 'string', 'trip_id': 'string', 'shape_id': 'string', 'route_id': 'string'}
df_stops = pd.read_csv(static_data_dir / 'stops.txt', dtype=id_dtypes)
df_stop_times = pd.read_csv(static_data_dir / 'stop_times.txt', dtype=id_dtypes)
df_trips = pd.read_csv(static_data_dir / 'trips.txt', dtype=id_dtypes)
df_shapes = pd.read_csv(static_data_dir / 'shapes.txt', dtype=id_dtypes)
print(f'Static GTFS snapshot: {static_data_dir}')

## Build the delay-proxy congestion engine

Take the latest publication for each `(trip instance, stop ID, stop sequence)`; greatest database ID breaks timestamp ties, otherwise first input row wins. Missing latest delays remain missing, with no fallback to older predictions.

Trip-start descriptors distinguish daily trips, including crossings of Vancouver midnight. Missing starts are excluded explicitly instead of approximately pairing visits. Each endpoint must match its own trip, direction, stop ID, and sequence in static GTFS. Opposite directions, different shapes, and express stop pairs remain separate. Adjacent scheduled sequences need not differ numerically by one.

Missing visits, missing arrival timestamps, obviously unfinished visits (proxy arrival after the recording window), and nonincreasing endpoint arrival times are audited and excluded. An absent intermediate stop is never bridged to invent a single-segment delta.

In [ ]:

time_bin_size = '5min'
max_congestion_age = pd.Timedelta(minutes=15)
congestion_smoothing_window = pd.Timedelta(minutes=50)
animation_window = pd.Timedelta(days=1)
animation_frame_duration_ms = 250
color_limit_seconds = 120

In [ ]:
delay_result = run_delay_congestion_engine(
    df_trip_updates,
    df_stops,
    df_stop_times,
    df_trips,
    df_shapes,
    route_id=route_id,
    trip_direction_id=trip_direction_id,
    time_bin_size=time_bin_size,
)
segments = delay_result.segments
df_traversals = delay_result.traversals
df_valid_traversals = df_traversals.loc[df_traversals.exclusion_reason.eq('included')]
df_congestion = delay_result.congestion
display(delay_result.audit)
print(f'Directed segments: {len(segments)}')
print(f'Usable traversals / candidate traversals: {len(df_valid_traversals):,} / {len(df_traversals):,}')
print(f'Observed segment bins: {len(df_congestion):,}')
print(f'Latest usable proxy crossing: {df_congestion.observed_at.max()}')
print(f'Recording end: {delay_result.observation_end}')

### Inspect coverage before interpreting congestion

Warnings and the audit are part of the analysis. Long realtime windows can contain trips absent from their cached static GTFS; these are not reassigned to a convenient reference trip. Inspect exclusions and the last usable crossing rather than assuming the whole recording period has coverage.

The legacy snapshot has no trip-update receipt timestamps, so historical availability uses publication time with an explicit warning. If future snapshots include receipts, the engine uses the later of publication and receipt. In both cases, a traversal also waits until its downstream proxy arrival.

In [ ]:
segment_coverage = df_valid_traversals.groupby('segment_id').size().rename('traversal_count')
display(segments[
    ['segment_id', 'trip_direction_id', 'shape_id', 'from_stop_name', 'to_stop_name']
].join(segment_coverage, on='segment_id'))
display(delay_result.stop_visits.loc[
    delay_result.stop_visits.exclusion_reason.ne('included'),
    ['trip_id', 'trip_start_timestamp', 'stop_id', 'stop_sequence', 'exclusion_reason'],
].head(10))
display(df_traversals.loc[
    df_traversals.exclusion_reason.ne('included'),
    ['trip_instance_key', 'from_stop_id', 'to_stop_id', 'exclusion_reason'],
].head(10))

## Example trip: accumulated delay versus segment contribution

Choose a trip instance below, or use the one with the most usable segment traversals. This is a retrospective explanation using final reported proxies, not an earlier point-in-time prediction. The upper panel shows total arrival delay; the lower panel shows only the delay gained or recovered on each incoming segment.

In [ ]:
example_trip_instance_key = None
if example_trip_instance_key is None:
    example_trip_instance_key = df_valid_traversals.groupby('trip_instance_key').size().idxmax()
example_visits = delay_result.stop_visits.loc[
    delay_result.stop_visits.trip_instance_key.eq(example_trip_instance_key)
    & delay_result.stop_visits.exclusion_reason.eq('included')
].sort_values('stop_sequence')
example_traversals = df_valid_traversals.loc[
    df_valid_traversals.trip_instance_key.eq(example_trip_instance_key)
].sort_values('to_stop_sequence')
if example_traversals.empty:
    raise ValueError('Selected example trip has no usable segment traversals.')
display(example_traversals[
    ['from_stop_id', 'to_stop_id', 'from_arrival_delay', 'to_arrival_delay',
     'delta_delay_seconds', 'crossed_at', 'available_at']
])

In [ ]:
example_figure = make_subplots(rows=2, cols=1, shared_xaxes=True)
example_figure.add_trace(go.Scatter(
    x=example_visits.stop_sequence, y=example_visits.arrival_delay / 60,
    mode='lines+markers', name='Accumulated arrival delay (proxy)',
), row=1, col=1)
example_figure.add_trace(go.Bar(
    x=example_traversals.to_stop_sequence,
    y=example_traversals.delta_delay_seconds / 60,
    name='Delay change on incoming segment',
), row=2, col=1)
example_figure.add_hline(y=0, line_dash='dash', row=1, col=1)
example_figure.add_hline(y=0, line_dash='dash', row=2, col=1)
example_figure.update_yaxes(title_text='Total delay (min)', row=1, col=1)
example_figure.update_yaxes(title_text='Segment change (min)', row=2, col=1)
example_figure.update_xaxes(title_text='Own-trip stop sequence', row=2, col=1)
example_figure.update_layout(title=f'Arrival-delay proxy decomposition: {example_trip_instance_key}', height=650)
example_figure.write_html(visuals_dir / 'route_97_delay_proxy_trip_example.html')
example_figure.show()

## Segment estimates and historical replay

Each completed traversal contributes **once** to its downstream arrival's five-minute UTC bin. The primary metric is the **median signed delay change in seconds**, not a clipped 0-1 speed index. Counts, spread, and positive/negative/unchanged shares reveal support and disagreement; a zero median can conceal buses experiencing opposite changes. A one-traversal bin is low-support evidence, and its standard deviation is undefined.

A final bin becomes available only after **all contributing endpoint evidence and proxy crossings**. No partial or future final bin is shown early. Replay selects the latest crossed bin already available; late backfilled older bins cannot replace newer crossings. Freshness is measured from the last contributing proxy crossing, not a delayed publication.

Raw and smoothed maps share the same display times and 15-minute provisional expiry. Smoothing is a trailing 50-minute mean of available actual bin medians, not repeated forward-filled frames. It never refreshes timestamps. Equivalent geometries display the freshest shape-specific source without averaging shapes.

The symmetric +/-120-second colour limit controls visual saturation only; full values remain in hover labels. Green means recovery, yellow means zero change, red means added delay, and grey means stale/unknown. These limits and windows are exploratory, not calibrated traffic thresholds.

In [ ]:
display(df_congestion[
    ['segment_id', 'shape_id', 'time_bin', 'median_delta_delay_seconds',
     'traversal_count', 'delta_delay_std_seconds', 'added_delay_share',
     'recovered_delay_share', 'unchanged_delay_share', 'available_at']
].head(10))
display(df_valid_traversals.delta_delay_seconds.describe())

In [ ]:
display_as_of = delay_result.observation_end
display_start = display_as_of - animation_window
display_times = pd.date_range(display_start, display_as_of, freq=time_bin_size)
display_times = display_times.union(pd.DatetimeIndex([display_as_of]))
raw_display = prepare_delay_congestion_snapshots(
    delay_result, display_times, max_age=max_congestion_age,
)
smoothed_display = prepare_delay_congestion_snapshots(
    delay_result, display_times, max_age=max_congestion_age,
    smoothing_window=congestion_smoothing_window,
)
latest_display = raw_display.loc[raw_display.as_of.eq(display_as_of)]
display(latest_display[
    ['segment_id', 'from_stop_name', 'to_stop_name', 'status',
     'last_delta_delay_seconds', 'observed_at', 'age', 'source_shape_id']
])

### Latest map

The map is anchored at the **recording end**, not silently moved to the last good observation. The selected legacy snapshot can therefore end with stale readings when later trips have no matching static GTFS. Change `display_as_of` explicitly above to examine an earlier period; grey segments are not evidence of free flow.

In [ ]:
latest_figure = make_delay_congestion_map(
    latest_display, delay_result.route_geometry,
    title=f'Route {route_id}, direction {trip_direction_id}: latest fresh delay-proxy changes',
    color_limit_seconds=color_limit_seconds,
)
latest_figure.write_html(visuals_dir / 'route_97_segment_congestion_via_delays.html')
latest_figure.show()

In [ ]:
raw_animation = make_delay_congestion_map(
    raw_display, delay_result.route_geometry,
    title=f'Route {route_id}: raw segment delay-proxy changes over {animation_window}',
    color_limit_seconds=color_limit_seconds,
    frame_duration_ms=animation_frame_duration_ms,
)
raw_animation.write_html(visuals_dir / 'route_97_segment_delay_changes_animation.html')
raw_animation.show()

In [ ]:
smoothed_animation = make_delay_congestion_map(
    smoothed_display, delay_result.route_geometry,
    title=f'Route {route_id}: trailing {congestion_smoothing_window} delay-proxy changes',
    color_limit_seconds=color_limit_seconds,
    frame_duration_ms=animation_frame_duration_ms,
)
smoothed_animation.write_html(visuals_dir / 'route_97_segment_delay_changes_smoothed_animation.html')
smoothed_animation.show()

## One shape-specific segment: signal and traversal support

Choose an engine segment ID from the coverage table. The default has the most usable traversals. Both panels span the full backtest, from the first recorded trip update through the recording end, using five-minute UTC bins. The upper panel is availability-aware replay for this exact shape, with stale/unknown gaps left blank. The lower panel is a **retrospective coverage diagnostic**: final distinct traversals per crossing bin, not repeated carried-forward frame counts and not an earlier-known feature. Hovering a count shows when its final evidence became available. Timestamps are displayed in Vancouver time.

In [ ]:
segment_id = int(segment_coverage.idxmax())
segment_id = 9
selected_segments = segments.loc[segments.segment_id.eq(segment_id)]
if selected_segments.empty:
    raise ValueError('Selected segment ID is not in this engine result.')
segment_bins = df_congestion.loc[df_congestion.segment_id.eq(segment_id)]
segment_start = pd.to_datetime(df_trip_updates.feed_timestamp, utc=True).min().floor(time_bin_size)
segment_end = delay_result.observation_end
segment_times = pd.date_range(segment_start, segment_end, freq=time_bin_size)
segment_times = segment_times.union(pd.DatetimeIndex([segment_end]))
selected_result = replace(delay_result, segments=selected_segments, congestion=segment_bins)
segment_raw = prepare_delay_congestion_snapshots(
    selected_result, segment_times, max_age=max_congestion_age,
)
segment_smooth = prepare_delay_congestion_snapshots(
    selected_result, segment_times, max_age=max_congestion_age,
    smoothing_window=congestion_smoothing_window,
)
segment_counts = segment_bins.loc[
    segment_bins.time_bin.between(segment_start, segment_end)
]
selected_segment = selected_segments.iloc[0]
segment_label = f'{selected_segment.from_stop_name} -> {selected_segment.to_stop_name} | shape {selected_segment.shape_id}'

In [ ]:
segment_figure = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3])
segment_figure.add_trace(go.Scatter(
    x=segment_raw.as_of.dt.tz_convert('America/Vancouver'),
    y=segment_raw.delta_delay_seconds / 60,
    mode='lines+markers', name='Raw median', connectgaps=False,
), row=1, col=1)
segment_figure.add_trace(go.Scatter(
    x=segment_smooth.as_of.dt.tz_convert('America/Vancouver'),
    y=segment_smooth.delta_delay_seconds / 60,
    mode='lines+markers', name='Trailing 50-minute mean', connectgaps=False,
), row=1, col=1)
segment_figure.add_trace(go.Bar(
    x=segment_counts.time_bin.dt.tz_convert('America/Vancouver'),
    y=segment_counts.traversal_count, name='Final traversals per crossing bin',
    customdata=segment_counts.available_at.astype('string'),
    hovertemplate='%{x}<br>Traversals: %{y}<br>Final evidence available (UTC): %{customdata}<extra></extra>',
), row=2, col=1)
segment_figure.add_hline(y=0, line_dash='dash', row=1, col=1)
segment_figure.update_yaxes(title_text='Delay change (min)', row=1, col=1)
segment_figure.update_yaxes(title_text='Distinct traversals', row=2, col=1)
segment_figure.update_xaxes(title_text='Time (Vancouver)', row=2, col=1)
segment_figure.update_xaxes(range=[
    segment_start.tz_convert('America/Vancouver'),
    segment_end.tz_convert('America/Vancouver'),
])
segment_figure.update_layout(title=f'Delay-proxy congestion and coverage: {segment_label}', height=700)
segment_figure.write_html(visuals_dir / 'route_97_segment_delay_changes_timeseries.html')
segment_figure.show()

In [ ]:
segment_figure = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3])
segment_figure.add_trace(go.Scatter(
    x=segment_raw.as_of.dt.tz_convert('America/Vancouver'),
    y=segment_raw.delta_delay_seconds / 60,
    mode='lines+markers', name='Raw median', connectgaps=False,
), row=1, col=1)
segment_figure.add_trace(go.Scatter(
    x=segment_smooth.as_of.dt.tz_convert('America/Vancouver'),
    y=segment_smooth.delta_delay_seconds / 60,
    mode='lines+markers', name='Trailing 50-minute mean', connectgaps=False,
), row=1, col=1)
segment_figure.add_trace(go.Bar(
    x=segment_counts.time_bin.dt.tz_convert('America/Vancouver'),
    y=segment_counts.traversal_count, name='Final traversals per crossing bin',
    customdata=segment_counts.available_at.astype('string'),
    hovertemplate='%{x}<br>Traversals: %{y}<br>Final evidence available (UTC): %{customdata}<extra></extra>',
), row=2, col=1)
segment_figure.add_hline(y=0, line_dash='dash', row=1, col=1)
segment_figure.update_yaxes(title_text='Delay change (min)', row=1, col=1)
segment_figure.update_yaxes(title_text='Distinct traversals', row=2, col=1)
segment_figure.update_xaxes(title_text='Time (Vancouver)', row=2, col=1)
segment_figure.update_xaxes(range=[
    segment_start.tz_convert('America/Vancouver'),
    segment_end.tz_convert('America/Vancouver'),
])
segment_figure.update_layout(title=f'Delay-proxy congestion and coverage: {segment_label}', height=700)
segment_figure.write_html(visuals_dir / 'route_97_segment_delay_changes_timeseries.html')
segment_figure.show()

## Delay propagation study

**Question:** when a segment gains unusually large delay, does it stay on that segment, or do *other buses* soon see extra delay on nearby segments? Queues spilling back move **upstream**; delay that rides with buses moves **downstream**.

### Why the first comparison was misleading

The first attempt compared the 50-minute smoothed display series of segments 9 and 10 at the same `as_of` time, split by hour of day. That comparison cannot separate propagation from coincidence:

1. **Shared daily pattern.** Both segments follow the daily cycle, so they correlate even when segment 10 is taken from a *different day* (table below).
2. **Repeated, overlapping readings.** With roughly one bus every 18 minutes, a five-minute bin almost always holds one traversal. Snapshots repeat that reading for up to 15 minutes, and 50-minute means overlap, so the scatter has far fewer independent points than it shows.
3. **No time lag or direction.** Spreading takes time, and the direction (upstream or downstream) is the answer we want.
4. **One pair, no uncertainty.** Hour-window correlations change sign on about 120-140 distinct readings, with no intervals.

In [ ]:
def smoothed_series(segment_id: int) -> pd.DataFrame:
    """Return one segment's smoothed display series indexed by as_of."""
    selected = replace(
        delay_result,
        segments=segments.loc[segments.segment_id.eq(segment_id)],
        congestion=df_congestion.loc[df_congestion.segment_id.eq(segment_id)],
    )
    snapshots = prepare_delay_congestion_snapshots(
        selected, segment_times, max_age=max_congestion_age,
        smoothing_window=congestion_smoothing_window,
    )
    return snapshots.set_index('as_of')[['delta_delay_seconds', 'observed_at']]


smooth_9, smooth_10 = smoothed_series(9), smoothed_series(10)
comparison_rows = []
for shift_days in (0, 1, 2, 7):
    moved = smooth_10.set_axis(smooth_10.index + pd.Timedelta(days=shift_days))
    paired = smooth_9.join(moved, lsuffix='_9', rsuffix='_10').dropna(
        subset=['delta_delay_seconds_9', 'delta_delay_seconds_10']
    )
    comparison_rows.append({
        'segment 10 taken from': 'same time' if shift_days == 0 else f'{shift_days} day(s) later',
        'correlation': paired.delta_delay_seconds_9.corr(paired.delta_delay_seconds_10),
        'paired frames': len(paired),
        'distinct reading pairs': len(paired[['observed_at_9', 'observed_at_10']].drop_duplicates()),
    })
display(pd.DataFrame(comparison_rows).round(3))

### Method

1. **Route axis.** Order the dominant shape's segments by stop sequence, with GTFS `shape_dist_traveled` positions and the scheduled **timepoint** flag of each upstream stop. Buses can be held at timepoints, which absorbs or adds delay.
2. **Anomalies.** For each traversal, `anomaly = delay change - usual delay change`. "Usual" is the median for the same segment, local hour, and weekday/weekend class from **other service days only**.
3. **Events.** An event is a traversal in the top 10% of its own segment's anomalies.
4. **Space-time response.** For each event, average the anomalies of **other buses** on segments up to four away, in time bins before and after the event. The **null** repeats this after randomly permuting service days within weekday/weekend class, at the same clock times. It keeps the daily pattern but breaks same-day links. *Excess* = observed minus null mean. p-values come from the permutation; 95% intervals resample service days.
5. **Same-bus check.** Does the bus that had the event keep gaining delay later on its own trip? Split by whether a timepoint lies between the segments.
6. **Labels.** In the 30 minutes after an event, a group *sticks* if its own segment stays elevated, and *spills upstream / spreads downstream* if neighbours one or two segments away gain at least 10 s with a Bonferroni-adjusted p < 0.05.

In [ ]:
propagation_shape_id = '51737'
event_quantile = 0.9
n_null_draws = 200
corridor_positions = {
    'Hwy 97 west (Westbank -> Queensway)': range(0, 5),
    'Harvey Ave (Queensway -> Cooper)': range(5, 9),
    'Hwy 97 N (Cooper -> UBCO)': range(9, 13),
}

ordered_segments = order_route_segments(
    segments, df_stop_times, df_trips, shape_id=propagation_shape_id,
)
position_to_segment = ordered_segments.set_index('position').segment_id
corridor_groups = {
    int(position_to_segment[position]): corridor
    for corridor, positions in corridor_positions.items()
    for position in positions
}
segment_groups = {
    int(row.segment_id): f'{row.segment_id}: {row.from_stop_name} -> {row.to_stop_name}'
    for row in ordered_segments.itertuples()
    if row.segment_id in set(df_valid_traversals.segment_id)
}
display(ordered_segments[
    ['position', 'segment_id', 'from_stop_name', 'to_stop_name',
     'from_shape_dist', 'length_shape_dist', 'from_is_timepoint']
].assign(corridor=ordered_segments.segment_id.map(corridor_groups)))

In [ ]:
anomalies = compute_delay_anomalies(df_traversals, ordered_segments)
print(f'Traversals on shape {propagation_shape_id}: {len(anomalies):,}')
print(f'Without enough other-day history for a baseline: {anomalies.anomaly_seconds.isna().sum():,}')
display(anomalies.groupby('segment_id').agg(
    traversals=('anomaly_seconds', 'size'),
    usual_change_median=('baseline_delta_delay_seconds', 'median'),
    anomaly_std=('anomaly_seconds', 'std'),
).round(1))

### Actual, expected and anomalous segment delay over the full backtest

Choose `comparison_segment_id` below (default **9**, matching the earlier segment example). All three series use the **same traversal timestamps** of this shape-specific segment over the entire recording window, with Vancouver local time on x and **seconds** on y:

- **Actual segment delay change:** downstream arrival delay minus upstream arrival delay, not the bus's total lateness.
- **Expected recurring delay (baseline):** the usual median delay change for that segment, local hour, and weekday/weekend class, estimated from other service days. This shows the repeated time-of-day pattern; leaving each service day out can make comparable days' baselines differ slightly.
- **Anomalous segment delay:** actual delay change minus the expected recurring baseline. Thus **actual = expected + anomalous**; anomalous zero means *usual*, not zero actual delay.

Unlike the earlier availability-aware, five-minute replay and trailing mean, this is a **retrospective, traversal-by-traversal comparison** with no additional smoothing or carried-forward readings. Missing baselines leave the expected and anomalous values blank while keeping the actual value. Lines break across gaps longer than 30 minutes. Use the range slider or drag to zoom, and the legend to toggle any series. Hover shows the trip and usual baseline. The interactive plot is also saved as HTML.

In [ ]:
comparison_segment_id = 4
comparison_traversals = anomalies.loc[
    anomalies.segment_id.eq(comparison_segment_id)
].sort_values('local_time', kind='stable')
if comparison_traversals.empty:
    raise ValueError('Selected segment has no traversals on the propagation study shape.')
if comparison_traversals.anomaly_seconds.isna().all():
    raise ValueError('Selected segment has no other-day baselines for a delay comparison.')

comparison_runs = comparison_traversals.local_time.diff().gt(pd.Timedelta(minutes=30)).cumsum()
comparison_start = pd.to_datetime(df_trip_updates.feed_timestamp, utc=True).min()
comparison_end = delay_result.observation_end
print(f'Segment {comparison_segment_id}: {len(comparison_traversals):,} traversals; '
      f'{comparison_traversals.anomaly_seconds.isna().sum():,} without an anomaly baseline')

In [ ]:
actual_anomaly_figure = go.Figure()
for column, label, colour in (
    ('delta_delay_seconds', 'Actual segment delay change', '#2a78d6'),
    ('baseline_delta_delay_seconds', 'Expected recurring delay (baseline)', '#2e8b57'),
    ('anomaly_seconds', 'Anomalous segment delay', '#e34948'),
):
    for number, (_, run) in enumerate(comparison_traversals.groupby(comparison_runs, sort=False)):
        actual_anomaly_figure.add_trace(go.Scatter(
            x=run.local_time, y=run[column], mode='lines+markers',
            name=label, legendgroup=column, showlegend=number == 0,
            line=dict(color=colour, width=1.5), marker=dict(size=4),
            connectgaps=False,
            customdata=run[['trip_instance_key', 'baseline_delta_delay_seconds']],
            hovertemplate=(
                '%{x|%Y-%m-%d %H:%M:%S}<br>Delay: %{y:+.1f} s'
                '<br>Usual delay change: %{customdata[1]:+.1f} s'
                '<br>Trip: %{customdata[0]}<extra>%{fullData.name}</extra>'
            ),
        ))
actual_anomaly_figure.add_hline(y=0, line_dash='dash', line_color='#777777')
actual_anomaly_figure.update_layout(
    title=(
        f'Route {route_id}: actual, expected and anomalous delay - segment {comparison_segment_id}'
        f'<br><sup>{segment_groups[comparison_segment_id]}</sup>'
    ),
    xaxis=dict(
        title='Time (Vancouver)', type='date', rangeslider=dict(visible=True),
        range=[timestamp.tz_convert('America/Vancouver').tz_localize(None)
               for timestamp in (comparison_start, comparison_end)],
    ),
    yaxis_title='Segment delay change / anomaly (seconds)',
    legend=dict(groupclick='togglegroup', itemclick='toggle', itemdoubleclick='toggleothers'),
    hovermode='x unified', height=650, plot_bgcolor='#fcfcfb',
)
actual_anomaly_figure.write_html(
    visuals_dir / f'route_97_segment_{comparison_segment_id}_actual_vs_anomalous_delay.html',
)
actual_anomaly_figure.show()

### Event-by-event delay growth and decay

Compare repeated patterns on **one segment at a time**. Each coloured curve is centred on a positive top-quantile traversal from the event definition above: **x = minutes from that traversal**, **y = anomalous segment delay in seconds** (observed delay change minus its usual other-day baseline). Negative x shows the build-up; positive x shows persistence, recovery, or further growth. Diamonds mark the event at time zero. The slope shows the rate of increase/decrease in seconds per minute; this is not a fitted exponential decay or total bus lateness.

Set `event_segment_ids` below to choose segments (defaults: 8, 9, 10). Set `event_service_date` to a service date such as `'2026-11-02'` to reduce clutter, or leave `None` for all days. Thresholds still use the full history. Adjust the before/after windows to inspect longer patterns. **Click a legend entry to hide/show an event; double-click to isolate it or restore the others.** Hover shows actual local time, trip, raw delay change, and baseline. Each plot is also saved as an interactive HTML file.

Points are **unsmoothed successive buses on the same segment**, including the anchor bus, not one bus travelling downstream. Zero means usual delay change, not zero absolute lateness. Missing baselines and gaps over `event_max_gap_minutes` break lines; windows never cross service days or pad unobserved recovery with zero. With roughly 18-minute headways, short-lived changes may be unobserved. Consecutive top-quantile traversals can belong to the same episode: overlapping curves are not independent incidents, and alignment on a high reading naturally favours apparent decay.

TODO: Delete this section later. The graph is useless

In [ ]:
event_segment_ids = [8, 9, 10]
event_service_date = None
event_before_minutes = 60
event_after_minutes = 90
event_max_gap_minutes = 30

In [ ]:
event_delay_figures = {}
for segment_id in event_segment_ids:
    event_delay_figures[segment_id] = make_delay_event_plot(
        anomalies, segment_id=segment_id,
        title=(
            f'Route {route_id}: delay growth and decay - segment {segment_id}'
            f'<br><sup>{segment_groups[segment_id]}</sup>'
        ),
        event_quantile=event_quantile, service_date=event_service_date,
        before_minutes=event_before_minutes, after_minutes=event_after_minutes,
        max_gap_minutes=event_max_gap_minutes,
    )
    event_delay_figures[segment_id].write_html(
        visuals_dir / f'route_97_segment_{segment_id}_delay_events.html',
    )
    event_delay_figures[segment_id].show()

### Space-time response

Each cell is the excess anomaly (seconds) seen by **other buses** on the segment `offset` positions away, within the lag bin, relative to the permuted-day null. `*` marks permutation p < 0.05. How to read it:

- excess only at offset 0 after the event: delay **sticks** (its persistence is how long offset 0 stays red);
- excess at negative offsets after the event: **upstream spillback**;
- excess at positive offsets after the event: **downstream spread**;
- excess at neighbours *before* the event, or at distant offsets: a shared cause such as a bad day or a corridor-wide incident, not spreading from the segment.

With hundreds of pooled events, differences of a few seconds can be starred. Judge size as well as stars.

In [ ]:
route_response = event_space_time_response(
    anomalies, event_quantile=event_quantile, n_null_draws=n_null_draws,
)
route_heatmap = make_propagation_heatmap(
    route_response,
    title=f'Route {route_id}, direction {trip_direction_id}: response of other buses to delay events',
)
route_heatmap.write_html(visuals_dir / 'route_97_delay_propagation_heatmap.html')
route_heatmap.show()

In [ ]:
corridor_response = event_space_time_response(
    anomalies, event_quantile=event_quantile, n_null_draws=n_null_draws,
    source_groups=corridor_groups,
)
for corridor in corridor_positions:
    make_propagation_heatmap(
        corridor_response, group=corridor,
        title='Response of other buses to delay events starting in this corridor',
    ).show()

### Stick or spread?

The labels use one window from 0 to 30 minutes after each event, plus the 30 minutes before it to check for neighbours that were already elevated. `spread_ratio` is the strongest neighbour excess divided by the segment's own excess. Per-segment rows have about 75 events each, so treat them as leads to investigate rather than conclusions.

In [ ]:
summary_kwargs = dict(
    event_quantile=event_quantile, n_null_draws=n_null_draws,
    offsets=range(-2, 3), lag_edges_minutes=(-30, 0, 30),
)
corridor_summary = summarize_propagation(
    event_space_time_response(anomalies, source_groups=corridor_groups, **summary_kwargs)
)
segment_summary = summarize_propagation(
    event_space_time_response(anomalies, source_groups=segment_groups, **summary_kwargs)
)
display(corridor_summary.round(3))
display(segment_summary.round(3))

### Same-bus carry-over

`excess_seconds` compares the event bus's own later (positive offset) or earlier (negative offset) anomaly with that of buses without an event on the source segment. Adjacent segments share one stop's arrival-delay proxy, so estimate errors at that stop push them in opposite directions. Offsets of two avoid that coupling. Rows crossing a timepoint show the effect of holding.

In [ ]:
same_bus = same_bus_propagation(
    anomalies, event_quantile=event_quantile, max_offset=2, source_groups=corridor_groups,
)
display(same_bus[
    ['group', 'offset', 'crosses_timepoint', 'event_count', 'excess_seconds',
     'excess_ci_low_seconds', 'excess_ci_high_seconds', 'spearman']
].round(2))

### Robustness

Repeat the corridor labels with a stricter event definition (top 5%), and with events only on segments that do not start at a timepoint. Holding at timepoints shapes those segments' delay changes.

In [ ]:
non_timepoint_groups = {
    segment: corridor for segment, corridor in corridor_groups.items()
    if not ordered_segments.set_index('segment_id').from_is_timepoint[segment]
}
robustness = {
    'top 10% events': corridor_summary,
    'top 5% events': summarize_propagation(event_space_time_response(
        anomalies, source_groups=corridor_groups, **{**summary_kwargs, 'event_quantile': 0.95},
    )),
    'non-timepoint sources': summarize_propagation(event_space_time_response(
        anomalies, source_groups=non_timepoint_groups, **summary_kwargs,
    )),
}
display(pd.DataFrame({
    name: table.set_index('group').apply(
        lambda row: f'{row.label} (self {row.self_excess_seconds:+.0f} s, '
                    f'up {row.upstream_excess_seconds:+.0f} s, '
                    f'down {row.downstream_excess_seconds:+.0f} s)',
        axis=1,
    )
    for name, table in robustness.items()
}))

### Time-space diagram

Each grey line is one trip moving up the route over time. Markers show segment anomalies. Delay that **spreads** forms red bands running diagonally across several trips; delay that **sticks** forms a horizontal red stripe on one segment. The default day has the most total added anomaly; set `diagram_date` to inspect another.

In [ ]:
anomalies.groupby('service_date').anomaly_seconds.apply(
        lambda values: values.clip(lower=0).sum()
    ).idxmax()

In [ ]:
diagram_date = pd.Timestamp(year=2026, month=10, day=26)
diagram_date

In [ ]:
#timestamp diagram data
diagram_date = None
diagram_date = pd.Timestamp(year=2026, month=10, day=26)

if diagram_date is None:
    diagram_date = anomalies.groupby('service_date').anomaly_seconds.apply(
        lambda values: values.clip(lower=0).sum()
    ).idxmax()
time_space = make_time_space_diagram(
    anomalies, ordered_segments, service_date=diagram_date,
    title=f'Route {route_id} time-space diagram, service day {pd.Timestamp(diagram_date):%Y-%m-%d}',
)
time_space.write_html(visuals_dir / 'route_97_delay_time_space_diagram.html')
time_space.show()

### Findings (`2026-11-09_14_days`, direction 0)

- **Delay mostly sticks to its segment.** After an event, other buses on the *same* segment see about +27 to +33 s of extra delay for 30 minutes and about +22 s at 30-60 minutes, fading by 60-90 minutes. Immediate neighbours show no consistent after-event response (within about +/-10 s).
- **Neighbour signals tend to come before the event, not after.** Extra delay one segment away in the preceding 15 minutes, and scattered cells three or four segments away, point to shared causes (bad days, corridor-wide incidents) rather than spreading.
- **Corridors.** Hwy 97 west and Harvey Ave *stick*: their own segments stay elevated for about 30-50 s. Hwy 97 N delays are short-lived (*no lasting signal*). With non-timepoint sources only, Harvey Ave shows modest **upstream spillback** (about +14 s).
- **Upstream spillback on Harvey Ave towards Orchard Park.** Parkinson -> Cooper (segment 8) is the clearest case: after its events, upstream neighbours gain about +49 s, consistent with queues backing up along Harvey Ave. Gordon -> Parkinson (segment 7) leans downstream, but not significantly.
- **Little same-bus carry-over.** Without a timepoint in between, the event bus gains about +6 to +16 s more on the next segment, and intervals include zero for two of the three corridors. It gains nothing two segments on. A similar excess on the segment *before* the event suggests trip-level conditions (crowding, a slow period) rather than delay being passed forward.
- **Timepoints reset delay.** Events on the segment leaving Queensway follow buses that ran about 72 s *earlier* than usual on the approach: they were held at the exchange, so these "events" are holding, not traffic. Crossing a timepoint after an event, buses recover (for example -22 s on Hwy 97 west).

### Limitations

Delay changes are arrival-to-arrival proxies that include dwell, holding, and prediction revisions, not GPS road speeds. Fifteen days with about 18-minute headways give roughly 75 events per segment, so per-segment labels are exploratory. Only direction 0 is analysed. Run the same cells for direction 1 by changing `trip_direction_id` and the shape ID. The p-values are not corrected across the heatmap's many cells.

## Sanity checks and limitations

Check exact endpoint subtraction, unique scheduled traversals, direction isolation, missing exclusions, and no evidence from after a replay timestamp. Package tests also exercise inherited lateness, early buses, duplicate updates, missing stops, midnight trips, shape variants, freshness boundaries, and smoothing.

Positive changes can arise from traffic, dwell, disruptions, or prediction revisions. Negative changes indicate recovery relative to the timetable, not guaranteed absolute free flow. Zero means no added delay relative to schedule, not necessarily an empty road. Schedule slack and holding remain confounders. Final reported arrival timestamps also determine bin placement and can still be imperfect, even after excluding obviously unfinished visits.

In [ ]:
assert df_valid_traversals.delta_delay_seconds.eq(
    df_valid_traversals.to_arrival_delay - df_valid_traversals.from_arrival_delay
).all()
assert not df_valid_traversals.duplicated(['trip_instance_key', 'to_stop_sequence']).any()
assert df_valid_traversals.trip_direction_id.eq(trip_direction_id).all()
assert df_traversals.loc[df_traversals.exclusion_reason.ne('included'), 'delta_delay_seconds'].isna().all()
known = raw_display.loc[raw_display.observed_at.notna()]
assert known.available_at.le(known.as_of).all()
assert raw_display.loc[raw_display.status.ne('fresh'), 'delta_delay_seconds'].isna().all()
print('Delay subtraction, coverage, direction, and historical availability checks passed.')